## Parking Lot Design 2
Start with requirements clarification — ask: multiple floors? vehicle types? pricing model? reservations? handicapped spots? This signals system-thinking before coding.

Core entities to name up front: ParkingLot, ParkingFloor, ParkingSlot, Vehicle, Ticket, ParkingRate. That's your skeleton.

Key design patterns to mention:

1. Singleton — ParkingLot, since there's one instance managing shared state
2. Strategy — ParkingRate interface so you can swap hourly/daily/monthly pricing without touching existing classes
3. Abstract class — Vehicle because all vehicles share fields, vs an interface which is for pure behavior

The concurrency question is almost always asked. Answer: synchronize findAndAssignSlot() at the floor level (not the lot level) using a ReentrantLock per floor — finer granularity means better throughput.

Extensibility points to mention proactively: EV charging slots (EVSlot extends ParkingSlot), reservations (add a Reservation entity with a reservedUntil timestamp), monthly passes (MonthlyRate implements ParkingRate).

The Live Simulation tab lets you demonstrate the flow end-to-end: park a vehicle → slot turns red → exit & pay → fee calculated. The Design Decisions tab has the key trade-off explanations you'd articulate verbally.

### Key design choices to articulate in the interview:

1. get_free_slot() marks is_free = False inside the lock before returning — this prevents two threads claiming the same slot between "find" and "assign"

2. slots_by_type: Dict[SlotType, List[ParkingSlot]] gives O(1) lookup for a specific slot type rather than scanning a flat list

3. SLOT_VEHICLE_COMPATIBILITY map cleanly encodes which vehicles fit which slot types — easy to extend

4. The build_default_lot() factory keeps construction separate from the Singleton, so tests can configure the lot independently

#### Classes & patterns implemented:
ComponentPatternNotesParkingLotSingletonThread-safe via double-checked lockingParkingRateStrategyHourlyRate, DailyRate, MonthlyRatePaymentMethodStrategyCash, Card, UPIVehicleAbstract classCar, Truck, Motorcycle, VanParkingFloor—Per-floor threading.Lock for concurrencyTicket—Entry/exit times, fee lifecycleEntrancePanel / ExitPanelFacadeThin wrappers over the lot